In [4]:
import pickle
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== 加载数据 =====
with open('data/20news_split.pkl', 'rb') as f:
    dataset = pickle.load(f)

vectorizer = CountVectorizer(max_features=5000)

X_train, y_train = dataset['train']
X_val, y_val = dataset['val']
X_test, y_test = dataset['test']

X_train = vectorizer.fit_transform(X_train).toarray()
X_val = vectorizer.transform(X_val).toarray()
X_test = vectorizer.transform(X_test).toarray()

# ===== Dataset =====
class SimpleDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(SimpleDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(SimpleDataset(X_val, y_val), batch_size=32)
test_loader = DataLoader(SimpleDataset(X_test, y_test), batch_size=32)

# ===== GRU模型 =====
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        x = x.unsqueeze(1)
        _, h = self.gru(x)
        return self.fc(h.squeeze(0))

model = GRUModel(5000).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ===== 训练 =====
def train():
    for epoch in range(5):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1}")
        evaluate(val_loader)

# ===== 评估 =====
def evaluate(loader):
    model.eval()
    preds, labels = [], []

    with torch.no_grad():
        for X, y in loader:
            X = X.to(device)
            outputs = model(X)
            preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
            labels.extend(y.numpy())

    acc = accuracy_score(labels, preds)
    print(f"Accuracy: {acc:.4f}")
    return acc

if __name__ == "__main__":
    train()
    print("Test:")
    evaluate(test_loader)

Epoch 1
Accuracy: 0.9074
Epoch 2
Accuracy: 0.9506
Epoch 3
Accuracy: 0.9383
Epoch 4
Accuracy: 0.9383
Epoch 5
Accuracy: 0.9383
Test:
Accuracy: 0.9778
